# Validating the agent against known-by-design ground truth

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kikolo3000/clinical-skill-architecture/blob/main/notebooks/02_synthetic_validation.ipynb)

The package ships **3 hand-crafted synthetic instances** with ground-truth codings. They are pulled from the 150-instance suite used in the paper, chosen to illustrate the three hardest things an automated coder has to do well:

| Instance                          | Tests                                                                  |
|-----------------------------------|-------------------------------------------------------------------------|
| `SYN_001_easy_hopelessness`       | The textbook easy positive — two clear HOP clauses with one distractor |
| `SYN_095_distractor`              | Distractor avoidance — one HOP clause buried in mundane chatter         |
| `SYN_110_denial`                  | False-negative avoidance — explicit denial of self-blame                |

Total runtime ~5 minutes, total cost ~$0.05.

In [ ]:
# Install (uncomment in Colab)
# %pip install "clinical-skill-architecture[eval,viz]"

import getpass, os
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Anthropic API key: ')

from csa import GGAgent
from csa.eval import evaluate_instance
from csa.eval.synthetic import list_demo_instances
import json, matplotlib.pyplot as plt

In [ ]:
# Load the 3 bundled demo instances
instances = list_demo_instances()
for inst in instances:
    gt = inst.ground_truth()
    print(f'{inst.id}: {len(gt)} fragment(s), {sum(len(g["codings"]) for g in gt)} ground-truth coding(s)')
    print('  transcript:')
    for line in inst.transcript().splitlines():
        print(f'    {line}')
    print()

In [ ]:
# Score all 3 with the agent (in parallel)
import asyncio
agent = GGAgent(model='claude-sonnet-4-6', max_concurrent=5)

async def score_all():
    return await asyncio.gather(*[agent.score_async(inst.transcript()) for inst in instances])

# Top-level await works in Jupyter and Colab kernels (they already run an event loop).
results = await score_all()

for inst, result in zip(instances, results):
    print(f'{inst.id}: total={result.total_depression:.2f} | flagged subs={[fr.screening.flagged_subscales for fr in result.fragments]}')

In [ ]:
# Side-by-side: ground truth vs prediction for each instance
def render_comparison(instance, result):
    print(f'━━━ {instance.id} ━━━')
    gt_records = instance.ground_truth()
    pred_records = [fr.to_dict() for fr in result.fragments]
    for gt_rec, pred_rec in zip(gt_records, pred_records):
        ev = evaluate_instance(gt_rec, pred_rec)
        gt_codings = gt_rec.get('codings', [])
        pred_codings = pred_rec.get('codings', [])
        print(f'  GT:   decision={gt_rec["screening"]["decision"]:24s}  {len(gt_codings)} coding(s)')
        print(f'  Pred: decision={pred_rec["screening"]["decision"]:24s}  {len(pred_codings)} coding(s)')
        print(f'  TP={ev["tp"]}  FN={ev["fn"]}  FP={ev["fp"]}')
        for c in gt_codings:
            print(f'    GT  [{c["subscale"]}/{c["sub_item"]}/w={c["weight"]}] "{c["clause"]}"')
        for c in pred_codings:
            print(f'    PR  [{c["subscale"]}/{c["sub_item"]}/w={c["weight"]}] "{c["clause"]}"')
    print()

for inst, result in zip(instances, results):
    render_comparison(inst, result)

In [ ]:
# Hierarchical accuracy: for matched clauses, how far down the tree does the agent go correctly?
from csa.eval.evaluator import _aggregate

instance_results = []
for inst, result in zip(instances, results):
    gt_records = inst.ground_truth()
    pred_records = [fr.to_dict() for fr in result.fragments]
    for gt_rec, pred_rec in zip(gt_records, pred_records):
        instance_results.append(evaluate_instance(gt_rec, pred_rec))

report = _aggregate(instance_results)
print(report)

In [ ]:
# Plot the hierarchical accuracy
labels = ['Detection\n(F1)', 'Subscale\ngiven\ndetected', 'Sub-item\ngiven\nsubscale',
          'Perspective\ngiven\nsub-item', 'Weight\ngiven\nperspective']
values = [
    report.detection_f1,
    report.hierarchical['subscale_given_detection'].accuracy,
    report.hierarchical['sub_item_given_subscale'].accuracy,
    report.hierarchical['perspective_given_sub_item'].accuracy,
    report.hierarchical['weight_given_perspective'].accuracy,
]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(labels, values, color='#2c7fb8')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy')
ax.set_title('Hierarchical attribution (each level conditional on the previous)')
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.2f}', ha='center', fontsize=10)
ax.axhline(1.0, color='grey', linestyle=':', linewidth=0.8)
plt.tight_layout(); plt.show()

total_cost = sum(r.estimated_cost_usd for r in results)
print(f'\nTotal cost across {len(instances)} instances: ${total_cost:.4f}')

## Want to run the full 150-instance suite?

```bash
python synthetic/generate_all.py                          # one-time, writes to synthetic/instances/
for d in synthetic/instances/*/; do
    id=$(basename "$d")
    mkdir -p runs/sonnet46/$id
    csa-score "$d/transcript.md" --json runs/sonnet46/$id/output.jsonl
done
```

Then:

```python
from csa.eval import evaluate_run
report = evaluate_run('synthetic/instances', 'runs/sonnet46')
print(report)              # Tables 2-3 of the paper
```

See `docs/reproduce_paper.md` for full reproduction instructions.